# CMPE 256 — Song Recommender System
## Starter Notebook

This notebook gives you everything you need to **get started** with the Kaggle competition:
- How to load the data files
- How to build and validate a prediction
- How to generate, download and submit a valid submission file

The model used here is the **Global Mean baseline**, the simplest possible predictor.  
Your job is to beat it.

---
> **Note:** EDA (Exploratory Data Analysis) is intentionally not included here. You are encouraged to explore the data yourself, understanding it is part of the assignment.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

# Load KAGGLE_USERNAME and KAGGLE_KEY from the .env file in this directory
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath("RS_HW2.ipynb")), ".env"))

DATA_DIR = os.path.abspath("/home/anthony/repos/cmpe256/data/raw") + "/" 
# os.path.join(os.path.dirname(os.path.abspath("/home/anthony/repos/cmpe256/data/raw")), "") + "/"
os.makedirs(DATA_DIR, exist_ok=True)

!pip install kaggle python-dotenv -q
!kaggle competitions download -c cmpe-256-song-recommender-system33 -p {DATA_DIR}
!unzip -q -o {DATA_DIR}/*.zip -d {DATA_DIR}


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


## Step 1 — Install & Import Libraries

In [2]:
# !pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from collections import defaultdict
from collections.abc import Callable 

print('Libraries ready.')

Libraries ready.


## Step 2 — Load the Data

Four files are provided:

| File | Description |
|------|-------------|
| `train.csv` | User-song pairs with ratings (1.0–10.0). Use this to train your model. |
| `test.csv` | User-song pairs **without** ratings. You must predict these. |
| `sample_submission.csv` | Shows the exact format your submission must follow. |
| `song_data.csv` | Optional song metadata (title, artist, album, year). Useful for hybrid models. |

In [3]:
print(f'Using data directory: {DATA_DIR}')
train      = pd.read_csv(DATA_DIR + 'train.csv',             encoding='latin-1')
test       = pd.read_csv(DATA_DIR + 'test.csv',              encoding='latin-1')
sample_sub = pd.read_csv(DATA_DIR + 'sample_submission.csv', encoding='latin-1')
song_data  = pd.read_csv(DATA_DIR + 'song_data.csv',         encoding='latin-1')

print('=== train.csv ===')
print(f'  Rows   : {len(train):,}')
print(f'  Columns: {list(train.columns)}')
display(train.head(3))

print('\n=== test.csv ===')
print(f'  Rows   : {len(test):,}')
print(f'  Columns: {list(test.columns)}')
display(test.head(3))

print('\n=== sample_submission.csv ===')
print(f'  Rows   : {len(sample_sub):,}')
print(f'  Columns: {list(sample_sub.columns)}')
display(sample_sub.head(3))

print('\n=== song_data.csv ===')
print(f'  Rows   : {len(song_data):,}')
print(f'  Columns: {list(song_data.columns)}')
display(song_data.head(3))

Using data directory: /home/anthony/repos/cmpe256/data/raw/
=== train.csv ===
  Rows   : 4,254,583
  Columns: ['user_id', 'song_id', 'rating']


,user_id,song_id,rating
0,1619409,1003008,10.00
1,1006031,1006608,5.50
2,1376103,1001728,4.25



=== test.csv ===
  Rows   : 1,063,646
  Columns: ['user_id', 'song_id']


,user_id,song_id
0,1553719,1000121
1,1089653,1001182
2,1536350,1008101



=== sample_submission.csv ===
  Rows   : 1,063,646
  Columns: ['RecommendationId', 'rating']


,RecommendationId,rating
0,1553719-1000121,5.37
1,1089653-1001182,5.37
2,1536350-1008101,5.37



=== song_data.csv ===
  Rows   : 198,724
  Columns: ['song_id', 'title', 'release', 'artist_name', 'year']


,song_id,title,release,artist_name,year
0,1191338,Silent Night,Monster Ballads X-Mas,Faster Pussy cat,2003
1,1178322,L'antarctique,Des cobras des tarentules,3 Gars Su'l Sofa,2007
2,1052052,Ethos of Coercion,Descend Into Depravity,Dying Fetus,2009


## Step 3 — Build a Validation Split

Before submitting to Kaggle, always validate your model **locally** using a held-out portion of `train.csv`.  
This gives you a reliable RMSE estimate without wasting Kaggle submissions.

We use an 80/20 split here.

In [4]:
df_train, df_val = train_test_split(train, test_size=0.2, random_state=42)

print(f'Training rows  : {len(df_train):,}')
print(f'Validation rows: {len(df_val):,}')

def rmse(y_true, y_pred):
    """Compute Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

Training rows  : 3,403,666
Validation rows: 850,917


## Step 4 — Baseline Model (Global Mean)

The simplest possible model: predict the **global average rating** for every user-song pair.

This is the official competition baseline. **Your goal is to beat this RMSE.**

In [5]:
# Compute global mean from training split only
global_mean = df_train['rating'].mean()
print(f'Global mean rating: {global_mean:.4f}')

# Predict the same value for every validation pair
val_preds = np.full(len(df_val), global_mean)
baseline_rmse = rmse(df_val['rating'], val_preds)
print(f'Baseline RMSE (validation): {baseline_rmse:.4f}')
print()
print('Beat this score on the public leaderboard to earn ranking points.')

Global mean rating: 5.3703
Baseline RMSE (validation): 1.7316

Beat this score on the public leaderboard to earn ranking points.


## Step 5 — Train on Full Data & Generate Predictions

Once you are happy with your model's validation RMSE, retrain it on the **full** `train.csv` (not just the 80% split) before generating your submission. This uses all available signal.

In [6]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
test_preds = np.full(len(test), global_mean_full)

print(f'Predictions shape : {test_preds.shape}')
print(f'Prediction range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'Prediction mean   : {test_preds.mean():.4f}')

Predictions shape : (1063646,)
Prediction range  : [5.370, 5.370]
Prediction mean   : 5.3703


## Matrix Factorization Data Handling

In [12]:
def MFDataHandling(data:pd.DataFrame):
    userIds = data['user_id'].unique()
    songIds = data['song_id'].unique()

    userIdsIndicesMap = {user:i for i,user in enumerate(userIds)}
    songIdsIndicesMap = {song:i for i,song in enumerate(songIds)}
    
    new_data = data.copy()
    new_data['user_id'] = new_data['user_id'].map(userIdsIndicesMap)
    new_data['song_id'] = new_data['song_id'].map(songIdsIndicesMap)
    
    return new_data
mfTrain = MFDataHandling(train)
mfTest = MFDataHandling(test)


In [22]:
mfTrain

,user_id,song_id,rating
0,0,0,10.00
1,1,1,5.50
2,2,2,4.25
3,3,3,9.25
4,4,4,5.00
...,...,...,...
4254578,58870,30323,5.00
4254579,642581,718,5.00
4254580,114140,10871,4.50
4254581,276387,5408,8.75


In [23]:
mfTest

,user_id,song_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
1063641,453211,54446
1063642,91572,13019
1063643,190040,36804
1063644,31722,14232


## Matrix Factorization with Stochatic Gradient Descent

In [47]:
from tqdm import tqdm
# Core of Matrix Factorization
# R (m x n)
# ^r_ui = = p_u * (q_i)^T
# p_u -> P (m x k)
# q_i -> Q (n x k)
# Goal R = P * Q^T
# E = r_ui - ^r_ui
# Objective function SSE + reg: sum((r_ui - ^r_ui)**2 + lambda(||p_u||**2 + ||q_i||**2))

# Find Q and P where SSE is minimized
class MatrixFactorization():
    # Initializes P an Q from R.
    # def __init__(self, R: np.ndarray, k: int, learning_rate: float, regularization_strength: float):
    def __init__(self, k: int, learning_rate: float, regularization_strength: float, epochs:int, baselines, global_mean):
        # Get training data's dimensions m users and n items.
        self.k = k
        self.P = None
        self.Q = None
        # Initialize the learning rate and lambda regularization.
        self.lr = learning_rate
        self.lam = regularization_strength
        self.epochs = epochs
        self.global_mean = global_mean
        self.user_baselines, self.song_baselines = baselines
    
    def __baseline_rmse(self, data: pd.DataFrame):
        # u_idx = data['user_id'].values
        # s_idx = data['song_id'].values
        ratings = data['rating'].values

        b_u = data['user_id'].map(self.user_baselines).fillna(0).to_numpy()
        b_i = data['song_id'].map(self.song_baselines).fillna(0).to_numpy()
        preds = self.global_mean + b_u + b_i
        preds = np.clip(preds,1,5)
        errors  = ratings - preds
        return np.sqrt(np.mean(errors**2))
    
    def __rmse(self, data: pd.DataFrame):
        u_idx = data['user_id'].values
        s_idx = data['song_id'].values
        ratings = data['rating'].values

        # b_u = data['user_id'].map(self.user_baselines).to_numpy()
        # b_i = data['song_id'].map(self.song_baselines).to_numpy()
        b_u = data['user_id'].map(self.user_baselines).fillna(0).to_numpy()
        b_i = data['song_id'].map(self.song_baselines).fillna(0).to_numpy()
        preds = self.global_mean + b_u + b_i + np.sum(self.P[u_idx] * self.Q[s_idx], axis=1)
        preds = np.clip(preds,1,5)
        errors  = ratings - preds
        return np.sqrt(np.mean(errors**2))

    def train(self, train_data: pd.DataFrame, val_data:pd.DataFrame):
        userIds = train_data['user_id'].values
        songIds = train_data['song_id'].values
        rating = train_data['rating'].values
        
        m = len(np.unique(userIds))
        n = len(np.unique(songIds))
        self.P = np.random.normal(0, 0.02,(m, self.k))
        self.Q = np.random.normal(0, 0.02,(n, self.k))
        # Outer loop stays clean
        for epoch in range(self.epochs):
            indices = np.random.permutation(len(userIds))
            
            # THE FIX: Wrap the inner loop. 
            # mininterval=2.0 prevents the "flicker" by limiting refreshes.
            # leave=False prevents the "new line" spam after each epoch.
            inner_pbar = tqdm(range(len(userIds)), 
                              desc=f"Epoch {epoch+1}/{self.epochs}", 
                              unit="samples", 
                              mininterval=2.0, 
                              leave=False)
            
            for a in inner_pbar:
                idx = indices[a]
                u, i, r = userIds[idx], songIds[idx], rating[idx]
                
                p_u = self.P[u].copy()
                q_i = self.Q[i].copy()
                r_ui = self.global_mean + self.user_baselines[u] + self.song_baselines[i] + p_u @ q_i
                e_ui = r - r_ui
                
                self.user_baselines[u] -= self.lr * (-2 * e_ui + 2*0.1 * self.user_baselines[u])
                self.song_baselines[i] -= self.lr * (-2 * e_ui + 2*0.1 * self.song_baselines[i])
                self.user_baselines[u] = np.clip(self.user_baselines[u], -5, 5)
                self.song_baselines[i] = np.clip(self.song_baselines[i], -5, 5)

                grad_p = (-2 * e_ui * q_i + 2 * self.lam * p_u)
                grad_q = (-2 * e_ui * p_u + 2 * self.lam * q_i)
                grad_p = np.clip(grad_p, -5, 5)
                grad_q = np.clip(grad_q, -5, 5)
                # In-place updates (using -= ensures self.P and self.Q change)
                self.P[u] -= self.lr * grad_p
                self.Q[i] -= self.lr * grad_q
                
            # Print a clean summary after each epoch
            train_rmse = self.__rmse(train_data)
            val_rmse = self.__rmse(val_data)
            baseline_rmse = self.__baseline_rmse(val_data)
            # epoch_rmse = np.sqrt(total_epoch_loss / len(userIds))
            # print(f"Epoch {epoch+1} Complete - Final Error: {epoch_rmse:.4f}")
            print(f"Epoch {epoch+1}: Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f} | Baseline RMSE: {baseline_rmse:.4f}")
            # if  prev_rmse and abs(val_rmse - prev_rmse) < 0.0001:
            #     print("="*8 +"EARLY STOPPING: No improvement!!!"+"="*8)
            #     break
            prev_rmse = val_rmse
    # Performs dot product on P & Q to obtain r_ui
    def predict(self,u,i):
        # ^r_ui = = p_u * (q_i)^T
        p_u = self.P[u]
        q_i = self.Q[i]
        r_ui = p_u @ q_i
        return r_ui




In [48]:

class Recommender():
    def __init__(self,data:pd.DataFrame, val:pd.DataFrame,k:int = 5,lr:float=1e-03,lam:float=0.01,epochs:int=15):
        self.data = data
        self.val = val
        
        # self.global_mean = data['rating'].mean()
        self.k = k
        self.lr=lr
        self.lam=lam

        def __get_globalMean_baselines_and_residuals(data: pd.DataFrame):
            global_mean = data['rating'].mean()
            user_means = data.groupby('user_id')['rating'].mean()
            user_baselines = (user_means - global_mean).to_dict()
            
            song_means = data.groupby('song_id')['rating'].mean()
            song_baselines = (song_means - global_mean).to_dict()
            # The residual is user-item component of the rating aka p @ q, which is what the MF model
            # is actually calculating.
            return global_mean, user_baselines, song_baselines, data
        

        (self.train_global_mean, 
         self.train_user_baselines, 
         self.train_song_baselines, 
         self.data) = __get_globalMean_baselines_and_residuals(self.data)
        
        (self.val_global_mean,
        self.val_user_baselines,
        self.val_song_baselines,
        self.val) = __get_globalMean_baselines_and_residuals(self.val)
        
        self.val_global_mean = self.train_global_mean
        self.val_user_baselines = self.train_user_baselines
        self.val_song_baselines = self.train_song_baselines
        

        self.mf = MatrixFactorization(k=self.k,
                                      learning_rate=self.lr,
                                      regularization_strength=self.lam,
                                      epochs=epochs,
                                      baselines=(self.train_user_baselines,self.train_song_baselines),
                                      global_mean=self.train_global_mean
                                      )
        self.mf.train(self.data,self.val)
    
    def predict(self,user,song):
        user_baseline = self.train_user_baselines.get(user,0)
        song_baseline = self.train_song_baselines.get(song,0)
        interaction = self.mf.predict(user,song)
        return global_mean + user_baseline + song_baseline + interaction

In [14]:
mfVal = MFDataHandling(df_val)
mfVal
u,s = list(mfVal[['user_id','song_id']].itertuples(index=False))[0]
u

0

In [26]:
print(set(mfVal['user_id']).issubset(set(mfTrain['user_id'])))

True


In [57]:
{"data":mfTrain,"val":mfVal,"k":8,"lr":5e-4,"lam":0.3,"epochs":25}
{"data":mfTrain,"val":mfVal,"k":5,"lr":5e-5,"lam":0.6,"epochs":25}
config =  {"data":mfTrain,"val":mfVal,
    "k": 10, 
    "lr": 5e-3,     # Slower updates
    "lam": 0.1,    # Much stricter regularization
    "epochs": 25
} 
config =  {"data":mfTrain,"val":mfVal,
    "k": 100, 
    "lr": 1e-3,     # Slower updates
    "lam": 0.04,    # Much stricter regularization
    "epochs": 20
} 

In [58]:
# recommender = Recommender(mfTrain,mfVal,k=8,lr=5e-4,lam=0.3,epochs=25)
recommender = Recommender(**config)

Epoch 1: Train RMSE: 1.7546 | Val RMSE: 1.8533 | Baseline RMSE: 1.8533


Epoch 2: Train RMSE: 1.7544 | Val RMSE: 1.8528 | Baseline RMSE: 1.8528


Epoch 3: Train RMSE: 1.7542 | Val RMSE: 1.8524 | Baseline RMSE: 1.8524


Epoch 4: Train RMSE: 1.7541 | Val RMSE: 1.8520 | Baseline RMSE: 1.8520


Epoch 5: Train RMSE: 1.7540 | Val RMSE: 1.8517 | Baseline RMSE: 1.8517


Epoch 6: Train RMSE: 1.7539 | Val RMSE: 1.8514 | Baseline RMSE: 1.8514


KeyboardInterrupt: 

In [96]:
mfTest

,user_id,song_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
1063641,453211,54446
1063642,91572,13019
1063643,190040,36804
1063644,31722,14232


In [104]:
mfTrain

,user_id,song_id,rating,residual
0,0,0,10.00,4.392582
1,1,1,5.50,0.080511
2,2,2,4.25,-1.442241
3,3,3,9.25,3.347556
4,4,4,5.00,0.181887
...,...,...,...,...
4254578,58870,30323,5.00,0.051343
4254579,642581,718,5.00,-1.512296
4254580,114140,10871,4.50,-0.645244
4254581,276387,5408,8.75,3.469036


In [114]:
recommender.predict(4,4)

np.float64(4.370966991265057)

In [103]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
# np.full(len(test), global_mean_full)
mf_test_preds = np.array([recommender.predict(u,s) for u,s in list(mfVal[['user_id','song_id']].itertuples(index=False))]) 
mf_baseline_rmse = rmse(mfVal['rating'], mf_test_preds)
print(f'Predictions shape : {mf_test_preds.shape}')
print(f'Prediction range  : [{mf_test_preds.min():.3f}, {mf_test_preds.max():.3f}]')
print(f'Prediction mean   : {mf_test_preds.mean():.4f}')
print(f'mf_baseline_rmse : {mf_baseline_rmse}, baseline_rmse : {baseline_rmse} {"success" if mf_baseline_rmse < baseline_rmse else "failed"}')
print(f'RMSE : {mf_baseline_rmse}')

Predictions shape : (850917,)
Prediction range  : [1.389, 16.227]
Prediction mean   : 6.1229
mf_baseline_rmse : 2.247189005931178, baseline_rmse : 1.7315939399182487 failed
RMSE : 2.247189005931178


## XGBOOST
To beat 1.7316

In [69]:
user_stats = mfTrain.groupby("user_id")['rating'].agg(['mean','count','std']).reset_index()
user_stats.columns = ['user_id','u_mean','u_count','u_std']
song_stats = mfTrain.groupby("song_id")['rating'].agg(['mean','count','std']).reset_index()
song_stats.columns = ['song_id','s_mean','s_count','s_std']
user_stats = user_stats.fillna(0)
song_stats = song_stats.fillna(0)

In [74]:
user_stats = mfTrain.groupby("user_id")['rating'].agg(['mean','count','std']).reset_index()
user_stats.columns = ['user_id','u_mean','u_count','u_std']
song_stats = mfTrain.groupby("song_id")['rating'].agg(['mean','count','std']).reset_index()
song_stats.columns = ['song_id','s_mean','s_count','s_std']
user_stats = user_stats.fillna(0)
song_stats = song_stats.fillna(0)

def get_xgb_features(df,user_stats,song_stats,global_mean):
    df_feat = df.merge(user_stats, on='user_id',how='left')
    df_feat = df_feat.merge(song_stats, on='song_id', how='left')

    df_feat['u_count'] = df_feat['u_count'].fillna(0)
    df_feat['s_count'] = df_feat['s_count'].fillna(0)
    df_feat['u_std'] = df_feat['u_std'].fillna(0)
    df_feat['s_std'] = df_feat['s_std'].fillna(0)

    df_feat['u_mean'] = df_feat['u_mean'].fillna(global_mean)
    df_feat['s_mean'] = df_feat['s_mean'].fillna(global_mean)

    df_feat['u_bias'] = df_feat['u_mean'] - global_mean
    df_feat['s_bias'] = df_feat['s_mean'] - global_mean

    df_feat = df_feat.fillna(global_mean)

    features = ['u_mean', 'u_count', 'u_std', 's_mean', 's_count', 's_std', 'u_bias', 's_bias']
    X = df_feat[features]
    y = df_feat.get('rating',None)

    return X,y
global_mean = mfTrain['rating'].mean()
X_train, y_train = get_xgb_features(mfTrain,user_stats,song_stats,global_mean)
X_val, y_val = get_xgb_features(mfVal,user_stats,song_stats,global_mean)

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=1000,       
    learning_rate=0.001, 
    max_depth=2,
    base_score=global_mean,
    tree_method='hist',
    eval_metric='rmse',
    early_stopping_rounds=5  
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

[0]	validation_0-rmse:1.73159
[10]	validation_0-rmse:1.73159
[13]	validation_0-rmse:1.73159


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",np.float64(5.370283585958953)
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",5
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.dataset

In [79]:
mfTest

,user_id,song_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
1063641,453211,54446
1063642,91572,13019
1063643,190040,36804
1063644,31722,14232


In [80]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
# test_preds = np.full(len(test), global_mean_full)
# mfTest['prediction']=.apply(lambda u,s: model.predict(u,s))
X_test, _ = get_xgb_features(mfTest, user_stats, song_stats, global_mean)
test_preds = np.clip(model.predict(X_test),1.0,10.0)

print(f'Predictions shape : {test_preds.shape}')
print(f'Prediction range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'Prediction mean   : {test_preds.mean():.4f}')

Predictions shape : (1063646,)
Prediction range  : [5.366, 5.377]
Prediction mean   : 5.3704


In [81]:
test_preds

array([5.3765054, 5.371369 , 5.3765054, ..., 5.3765054, 5.3765054,
       5.3660927], shape=(1063646,), dtype=float32)

## Step 6 — Build & Validate the Submission File

The submission format requires:
- Column `RecommendationId` — formatted as `user_id-song_id`
- Column `rating` — your predicted rating, clipped to [1.0, 10.0]

We validate the format against `sample_submission.csv` before saving.

In [82]:
# ── Build submission dataframe ────────────────────────────────────────────────
submission = pd.DataFrame({
    'RecommendationId': test['user_id'].astype(str) + '-' + test['song_id'].astype(str),
    'rating': np.clip(test_preds, 1.0, 10.0)   # always clip to valid range
})

# ── Format validation ─────────────────────────────────────────────────────────
print('=== Submission Validation ===')

# 1. Column names match
assert list(submission.columns) == list(sample_sub.columns), \
    f'Column mismatch: got {list(submission.columns)}, expected {list(sample_sub.columns)}'
print('✓ Column names match sample_submission.csv')

# 2. Row count matches test.csv
assert len(submission) == len(test), \
    f'Row count mismatch: got {len(submission)}, expected {len(test)}'
print(f'✓ Row count matches test.csv ({len(submission):,} rows)')

# 3. No missing values
assert submission.isnull().sum().sum() == 0, 'Submission contains NaN values!'
print('✓ No missing values')

# 4. Ratings within valid range
assert submission['rating'].between(1.0, 10.0).all(), 'Some ratings are outside [1.0, 10.0]!'
print('✓ All ratings within valid range [1.0, 10.0]')

# 5. RecommendationId format looks correct
id_check = submission['RecommendationId'].str.match(r'^\d+-\d+$').all()
assert id_check, 'Some RecommendationId values have unexpected format!'
print('✓ RecommendationId format is correct (user_id-song_id)')

print()
print('All checks passed. Ready to submit!')
print()
display(submission.head(5))

=== Submission Validation ===
✓ Column names match sample_submission.csv
✓ Row count matches test.csv (1,063,646 rows)
✓ No missing values
✓ All ratings within valid range [1.0, 10.0]
✓ RecommendationId format is correct (user_id-song_id)

All checks passed. Ready to submit!



,RecommendationId,rating
0,1553719-1000121,5.376505
1,1089653-1001182,5.371369
2,1536350-1008101,5.376505
3,1619708-1017052,5.376505
4,1048204-1013148,5.366296


## Step 7 — Save & Download the Submission File

In [83]:
OUTPUT_FILE = DATA_DIR + 'solution.csv'
submission.to_csv(OUTPUT_FILE, index=False)
print(f'Saved to: {OUTPUT_FILE}')

Saved to: /home/anthony/repos/cmpe256/data/raw/solution.csv


## Step 8 — Check if solution file format matching with sample solution file

In [84]:
sub    = pd.read_csv(OUTPUT_FILE)
sample = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print('=== Your Submission ===')
print('Columns:', sub.columns.tolist())
print('Rows:   ', len(sub))
print('dtypes:')
print(sub.dtypes)
print()
print('Preview:')
print(sub.head(3))

print()
print('=== Sample Submission ===')
print('Columns:', sample.columns.tolist())
print('Rows:   ', len(sample))
print('dtypes:')
print(sample.dtypes)
print()
print('Preview:')
print(sample.head(3))

print()
print('=== Validation Checks ===')
print('Column names match:       ', sub.columns.tolist() == sample.columns.tolist())
print('Row counts match:         ', len(sub) == len(sample))
print('No NaN in rating:         ', sub[sub.columns[1]].isna().sum() == 0)
print('Ratings in range [1, 10]: ', sub[sub.columns[1]].between(1, 10).all())
print('IDs match sample:         ', set(sub[sub.columns[0]]) == set(sample[sample.columns[0]]))
print('ID format correct:        ', sub[sub.columns[0]].str.contains(r'^\d+-\d+$').all())

=== Your Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId    rating
0  1553719-1000121  5.376505
1  1089653-1001182  5.371369
2  1536350-1008101  5.376505

=== Sample Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId  rating
0  1553719-1000121    5.37
1  1089653-1001182    5.37
2  1536350-1008101    5.37

=== Validation Checks ===
Column names match:        True
Row counts match:          True
No NaN in rating:          True
Ratings in range [1, 10]:  True
IDs match sample:          True
ID format correct:         True


## Step 9 — Submit to kaggle directly from google colab (or upload solution csv manually)

In [85]:
!kaggle competitions submit \
    -c cmpe-256-song-recommender-system33 \
    -f {OUTPUT_FILE} \
    -m "baseline kaggle competition submission"

print('Submitted - check leaderboard at:')
print('https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload
Submitted - check leaderboard at:
https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard
